# M1 Notebook 12 — Probability Spaces and Simulation

**Notebook ID:** M1_N12  
**Status:** Runnable first edition  
**Random seed:** 42

> Probability provides the mathematical language for uncertainty. Simulation turns probabilistic models into observable numerical experiments.


## 1. Learning objectives

1. Define sample spaces, events, and probabilities.
2. Apply the probability axioms.
3. Compute unions, intersections, complements, and conditional probabilities.
4. Distinguish independence from mutual exclusivity.
5. Estimate probabilities empirically.
6. Demonstrate the law of large numbers.
7. Use Monte Carlo simulation for uncertainty analysis.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.probability import (
    FiniteProbabilitySpace,
    empirical_probability,
    law_of_large_numbers_path,
    monte_carlo_expectation,
    simulate_bernoulli,
    simulate_categorical,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## 2. Probability space

A probability space is

\[
(\Omega,\mathcal F,P),
\]

where:

- \(\Omega\) is the sample space;
- \(\mathcal F\) is a collection of events;
- \(P\) assigns probabilities to events.


## 3. Probability axioms

For events \(A\) and \(B\):

\[
P(A)\ge0,
\qquad
P(\Omega)=1.
\]

If \(A\cap B=\varnothing\), then

\[
P(A\cup B)=P(A)+P(B).
\]


In [ ]:
coin = FiniteProbabilitySpace({
    "H": 0.5,
    "T": 0.5,
})

assert np.isclose(coin.probability({"H"}), 0.5)
assert np.isclose(coin.probability({"H", "T"}), 1.0)

{
    "P(H)": coin.probability({"H"}),
    "P(T)": coin.probability({"T"}),
    "P(Ω)": coin.probability({"H", "T"}),
}


## 4. Two-coin sample space

\[
\Omega=\{HH,HT,TH,TT\}.
\]


In [ ]:
two_coins = FiniteProbabilitySpace({
    "HH": 0.25,
    "HT": 0.25,
    "TH": 0.25,
    "TT": 0.25,
})

first_head = {"HH", "HT"}
second_head = {"HH", "TH"}
at_least_one_head = {"HH", "HT", "TH"}

results = {
    "P(first head)": two_coins.probability(first_head),
    "P(second head)": two_coins.probability(second_head),
    "P(at least one head)": two_coins.probability(at_least_one_head),
}
results


## 5. Inclusion–exclusion

\[
P(A\cup B)
=
P(A)+P(B)-P(A\cap B).
\]


In [ ]:
A = first_head
B = second_head

lhs = two_coins.probability(A | B)
rhs = (
    two_coins.probability(A)
    + two_coins.probability(B)
    - two_coins.probability(A & B)
)

assert np.isclose(lhs, rhs)
lhs, rhs


## 6. Conditional probability

\[
P(A\mid B)
=
\frac{P(A\cap B)}{P(B)},
\qquad P(B)>0.
\]


In [ ]:
conditional = two_coins.conditional_probability(
    first_head,
    second_head,
)

assert np.isclose(conditional, 0.5)
conditional


## 7. Independence

Events \(A\) and \(B\) are independent when

\[
P(A\cap B)=P(A)P(B).
\]


In [ ]:
is_independent = two_coins.independent(
    first_head,
    second_head,
)

assert is_independent
is_independent


Mutually exclusive events cannot occur together. Independent events do not influence each other's probabilities. These are different concepts.


## 8. Empirical probability

In [ ]:
observations = ["H", "T", "H", "H", "T", "H"]

empirical = empirical_probability(
    observations,
    {"H"},
)

assert np.isclose(empirical, 4/6)
empirical


## 9. Bernoulli simulation

A Bernoulli random variable takes values \(0\) and \(1\) with

\[
P(X=1)=p.
\]


In [ ]:
samples = simulate_bernoulli(
    probability=0.7,
    size=20,
    seed=42,
)

samples


## 10. Law of large numbers

As the sample size grows, the sample mean tends to stabilize near the expected value.


In [ ]:
path = law_of_large_numbers_path(
    probability=0.7,
    size=10_000,
    seed=42,
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(path)
ax.axhline(0.7, linestyle="--")
ax.set_xlabel("Number of simulations")
ax.set_ylabel("Cumulative sample mean")
ax.set_title("Law of Large Numbers for Bernoulli Trials")
plt.show()


In [ ]:
assert np.isclose(path[-1], 0.7, atol=0.02)
path[-1]


## 11. Categorical simulation

In [ ]:
weather = simulate_categorical(
    outcomes=["Dry", "Normal", "Wet"],
    probabilities=[0.25, 0.50, 0.25],
    size=10_000,
    seed=42,
)

frequencies = pd.Series(weather).value_counts(normalize=True).sort_index()
frequencies


## 12. Monte Carlo expectation

For samples \(X_1,\ldots,X_n\),

\[
\mathbb E[f(X)]
\approx
\frac{1}{n}
\sum_{i=1}^{n}f(X_i).
\]


In [ ]:
rng = np.random.default_rng(42)

estimate = monte_carlo_expectation(
    function=lambda x: x**2,
    sampler=lambda n: rng.uniform(0.0, 1.0, size=n),
    samples=200_000,
)

assert np.isclose(estimate, 1/3, atol=0.003)
estimate


## 13. Convergence of Monte Carlo estimates

In [ ]:
sample_sizes = [100, 1_000, 10_000, 100_000]
estimates = []

for n in sample_sizes:
    rng_local = np.random.default_rng(42)
    values = rng_local.uniform(0.0, 1.0, size=n)
    estimates.append({
        "samples": n,
        "estimate": np.mean(values**2),
        "absolute_error": abs(np.mean(values**2) - 1/3),
    })

pd.DataFrame(estimates)


## 14. Statistics interpretation

Probability spaces provide the foundation for random variables, distributions, expectation, variance, inference, and uncertainty quantification.


## 15. AI interpretation

Probability supports:

- probabilistic classification;
- Bayesian learning;
- generative models;
- reinforcement learning;
- uncertainty-aware predictions;
- Monte Carlo methods;
- stochastic optimization.


## 16. Decision Intelligence case — Drought contingency simulation

Suppose the coming season may be Dry, Normal, or Wet. Each state implies a modeled emergency cost.


In [ ]:
states = np.array(["Dry", "Normal", "Wet"])
probabilities = np.array([0.30, 0.50, 0.20])
costs = {
    "Dry": 120.0,
    "Normal": 35.0,
    "Wet": 50.0,
}

simulated_states = simulate_categorical(
    states,
    probabilities,
    size=100_000,
    seed=7,
)

simulated_costs = np.array([
    costs[state]
    for state in simulated_states
])

summary = {
    "expected_cost": simulated_costs.mean(),
    "cost_standard_deviation": simulated_costs.std(ddof=1),
    "probability_cost_exceeds_100": np.mean(simulated_costs > 100),
}
summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
pd.Series(simulated_costs).value_counts().sort_index().plot(
    kind="bar",
    ax=ax,
)
ax.set_xlabel("Modeled emergency cost")
ax.set_ylabel("Simulation count")
ax.set_title("Drought Contingency Cost Simulation")
plt.show()


### Interpretation

Simulation translates assumptions about uncertain states into distributions of outcomes. It does not make the assumptions true. Decision use requires evidence-based probabilities, sensitivity analysis, model validation, and explicit treatment of rare severe events.


## 17. Engineering notes

- Random seeds support reproducibility.
- Pseudo-random numbers are deterministic given the generator and seed.
- Monte Carlo uncertainty decreases slowly, approximately with the square root of sample size.
- Rare-event probabilities may require specialized methods.
- Simulation output should include uncertainty intervals, not only averages.


## 18. Common errors

- Confusing empirical frequency with exact probability.
- Treating independence as mutual exclusivity.
- Conditioning on an event with zero probability.
- Using arbitrary probabilities without evidence.
- Reporting only mean simulated outcomes.
- Assuming more simulations fix a misspecified model.


## 19. Exercises

### Level A
Define sample space, event, and probability.

### Level B
Verify inclusion–exclusion for three events.

### Level C
Simulate a categorical risk model and compare empirical with theoretical probabilities.

### Capstone
Develop a reproducible contingency simulation, document the probability model, estimate expected losses and tail risks, and test sensitivity to alternative assumptions.


## 20. Key insight

Probability formalizes uncertainty. Simulation makes probabilistic assumptions computationally observable. Together, they allow analysts to examine not only expected outcomes but also variability, tail risks, and decision consequences.
